In [22]:
# imports and broadcasting helper function 

In [23]:
import numpy as np
from sklearn.datasets import fetch_openml, make_moons
from sklearn.model_selection import train_test_split

#it broadcasts the smaller matrix
def unbroadcast(grad, original_shape):
    ndims_added = grad.ndim - len(original_shape)
    for _ in range(ndims_added):
        grad = grad.sum(axis=0) # sum outs new dimensions
    
    for i, dim in enumerate(original_shape):
        if dim == 1:
            grad = grad.sum(axis=i, keepdims=True)
            
    return grad

In [24]:
# autograd tensor class

In [25]:
class Tensor:
    def __init__(self, data, requires_grad=False, _children=()):
        self.data = np.array(data, dtype=np.float32)
        self.requires_grad = requires_grad
        self.grad = np.zeros_like(self.data) if requires_grad else None
        # store chain rule math operation
        self._backward = lambda: None
        # stores parent tensors
        self._prev = set(_children)

    def __repr__(self):
        return f"Tensor({self.data}, requires_grad={self.requires_grad})"

    def zero_grad(self):
        # clear old gradients for new training
        if self.requires_grad:
            self.grad = np.zeros_like(self.data)

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, _children=(self, other))
        out.requires_grad = self.requires_grad or other.requires_grad

        def _backward():
            # The derivative of addition is 1. Local gradient flows straight through
            if self.requires_grad:
                self.grad += unbroadcast(out.grad, self.data.shape)
            if other.requires_grad:
                other.grad += unbroadcast(out.grad, other.data.shape)
        out._backward = _backward # attach math logic to the tensor
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, _children=(self, other))
        out.requires_grad = self.requires_grad or other.requires_grad

        def _backward():
            # The derivative of A * B with respect to A is B
            if self.requires_grad:
                self.grad += unbroadcast(other.data * out.grad, self.data.shape)
            if other.requires_grad:
                other.grad += unbroadcast(self.data * out.grad, other.data.shape)
        out._backward = _backward
        return out

    def matmul(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(np.matmul(self.data, other.data), _children=(self, other))
        out.requires_grad = self.requires_grad or other.requires_grad

        def _backward():
            # For matrix multiplication Y = XW, dL/dX = dL/dY * W.T
            if self.requires_grad:
                self.grad += np.matmul(out.grad, other.data.T)
            if other.requires_grad:
                other.grad += np.matmul(self.data.T, out.grad)
        out._backward = _backward
        return out

    def exp(self):
        out = Tensor(np.exp(self.data), _children=(self,))
        out.requires_grad = self.requires_grad

        def _backward():
            if self.requires_grad:
                self.grad += unbroadcast(out.data * out.grad, self.data.shape)
        out._backward = _backward
        return out

    def log(self):
        # Adding 1e-15 prevents log(0) which causes a mathematical error (NaN)
        out = Tensor(np.log(self.data + 1e-15), _children=(self,))
        out.requires_grad = self.requires_grad

        def _backward():
            if self.requires_grad:
                self.grad += unbroadcast((1.0 / (self.data + 1e-15)) * out.grad, self.data.shape)
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + np.exp(-self.data))
        out = Tensor(s, _children=(self,))
        out.requires_grad = self.requires_grad

        def _backward():
            if self.requires_grad:
                 # The derivative of sigmoid(x) is sigmoid(x) * (1 - sigmoid(x))
                out_grad_factor = s * (1.0 - s)
                self.grad += unbroadcast(out_grad_factor * out.grad, self.data.shape)
        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, _children=(self,))
        out.requires_grad = self.requires_grad

        def _backward():
            if self.requires_grad:
                # The derivative of tanh(x) is 1 - tanh(x)^2
                out_grad_factor = 1.0 - t**2
                self.grad += unbroadcast(out_grad_factor * out.grad, self.data.shape)
        out._backward = _backward
        return out

    def backward(self):
        # Base case: The gradient of the final loss with respect to itself is 1.0
        if self.grad is None:
            self.grad = np.ones_like(self.data)

        # Topological Sort: Ensures a node only calculates its backward pass AFTER
        # all nodes that depend on it have finished calculating their gradients.
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Initialize all intermediate gradients to zero before accumulating
        for v in topo:
            if v.requires_grad and v.grad is None:
                v.grad = np.zeros_like(v.data)

        # Apply the chain rule backwards through the sorted graph
        for v in reversed(topo):
            v._backward()

In [26]:
# architecture layers

In [28]:
class Module:
    # Base class to collect parameters across all layers in a network
    def parameters(self):
        return []

    def zero_grad(self):
        for p in self.parameters():
            p.zero_grad()

class Linear(Module):
    def __init__(self, in_features, out_features):
        # Scales random weights so outputs don't explode or vanish
        self.weight = Tensor(np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features), requires_grad=True)
        self.bias = Tensor(np.zeros((1, out_features)), requires_grad=True)

    def __call__(self, x):
        return x.matmul(self.weight) + self.bias

    def parameters(self):
        return [self.weight, self.bias]

class ReLU(Module):
    def __call__(self, x):
        # Forward: Replaces all negative numbers with 0
        out = Tensor(np.maximum(0, x.data), _children=(x,))
        out.requires_grad = x.requires_grad

        def _backward():
            if x.requires_grad:
                # Backward: Gradient passes through only where output was > 0
                x.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

class Flatten(Module):
    def __call__(self, x):
        # Reshapes 4D image batches (Batch, Channels, Height, Width) into 2D matrices for Linear layers
        out = Tensor(x.data.reshape(x.data.shape[0], -1), _children=(x,))
        out.requires_grad = x.requires_grad

        def _backward():
            if x.requires_grad:
                # Restores the gradient back to the original 4D shape for the Conv layer behind it
                x.grad += out.grad.reshape(x.data.shape)
        out._backward = _backward
        return out

class Conv2D(Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels # Number of filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        scale = np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.weight = Tensor(np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * scale, requires_grad=True)
        self.bias = Tensor(np.zeros((out_channels, 1)), requires_grad=True)

    def __call__(self, x):
        x_data = x.data
        if self.padding > 0:
            # Adds borders of zeros around the image to prevent size reduction
            x_data = np.pad(x_data, ((0,0), (0,0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')
            
        N, C, H, W = x_data.shape
        F, _, HH, WW = self.weight.data.shape

        # Calculate the size of the new output images
        out_H = (H - HH) // self.stride + 1
        out_W = (W - WW) // self.stride + 1
        out_data = np.zeros((N, F, out_H, out_W), dtype=np.float32)

        # Sliding Window operation
        for n in range(N):
            for f in range(F):
                for h in range(out_H):
                    for w in range(out_W):
                        h_start = h * self.stride
                        h_end = h_start + HH
                        w_start = w * self.stride
                        w_end = w_start + WW
                        
                        patch = x_data[n, :, h_start:h_end, w_start:w_end]
                        # Multiply the image patch by the filter weights and sum it
                        out_data[n, f, h, w] = np.sum(patch * self.weight.data[f]) + self.bias.data[f, 0]
                        
        out = Tensor(out_data, _children=(x, self.weight, self.bias))
        out.requires_grad = x.requires_grad or self.weight.requires_grad or self.bias.requires_grad

        def _backward():
            if x.requires_grad or self.weight.requires_grad or self.bias.requires_grad:
                dx_padded = np.zeros_like(x_data)
                for n in range(N):
                    for f in range(F):
                        for h in range(out_H):
                            for w in range(out_W):
                                h_start = h * self.stride
                                h_end = h_start + HH
                                w_start = w * self.stride
                                w_end = w_start + WW

                                # Accumulate gradients back into the specific patch pixels and weights
                                if self.weight.requires_grad:
                                    self.weight.grad[f] += x_data[n, :, h_start:h_end, w_start:w_end] * out.grad[n, f, h, w]
                                if self.bias.requires_grad:
                                    self.bias.grad[f, 0] += out.grad[n, f, h, w]
                                if x.requires_grad:
                                    dx_padded[n, :, h_start:h_end, w_start:w_end] += self.weight.data[f] * out.grad[n, f, h, w]
                                    
                if x.requires_grad:
                    # Strip away the padding from the gradients before passing backwards
                    if self.padding > 0:
                        x.grad += dx_padded[:, :, self.padding:-self.padding, self.padding:-self.padding]
                    else:
                        x.grad += dx_padded
                        
        out._backward = _backward
        return out

    def parameters(self):
        return [self.weight, self.bias]

In [29]:
# loss functions and optimizzers

In [30]:
def cross_entropy_loss(logits, targets):
    # 1. Softmax: Convert raw network outputs into percentages
    exps = np.exp(logits.data - np.max(logits.data, axis=1, keepdims=True))
    probs = exps / np.sum(exps, axis=1, keepdims=True)
    m = targets.shape[0]

    # 2. Negative Log-Likelihood: Punish the network if the correct class has a low percentage
    log_likelihood = -np.log(probs[range(m), targets])
    loss_val = np.sum(log_likelihood) / m
    
    out = Tensor(loss_val, _children=(logits,))
    out.requires_grad = logits.requires_grad
    
    def _backward():
        if logits.requires_grad:
            # Mathematical shortcut: The derivative of Softmax + CrossEntropy is simply
            dlogits = probs.copy()
            dlogits[range(m), targets] -= 1
            dlogits = dlogits / m
            logits.grad += unbroadcast(dlogits * out.grad, logits.data.shape)
            
    out._backward = _backward
    return out

class SGD:
    def __init__(self, parameters, lr=0.01):
        self.parameters = parameters
        self.lr = lr

    def step(self):
        for p in self.parameters:
            if p.requires_grad and p.grad is not None:
                # Moves the weights slightly in the opposite direction of the gradient to reduce loss
                p.data -= self.lr * p.grad

class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.t = 0
        # Tracks the running average of past gradients
        self.m = [np.zeros_like(p.data) for p in self.parameters]
        # Tracks the running average of squared gradients
        self.v = [np.zeros_like(p.data) for p in self.parameters]

    def step(self):
        self.t += 1
        for i, p in enumerate(self.parameters):
            if p.requires_grad and p.grad is not None:
                # Update running averages
                self.m[i] = self.beta1 * self.m[i] + (1.0 - self.beta1) * p.grad
                self.v[i] = self.beta2 * self.v[i] + (1.0 - self.beta2) * (p.grad ** 2)

                # Bias correction for the first few steps
                m_hat = self.m[i] / (1.0 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1.0 - self.beta2 ** self.t)

                # Update weights: Use momentum, but slow down for parameters that bounce around wildly
                p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

In [31]:
# two moon dataset

In [32]:
X_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=42)
X_moons = X_moons.astype(np.float32)

class MoonMLP(Module):
    def __init__(self):
        self.fc1 = Linear(2, 16)
        self.fc2 = Linear(16, 16)
        self.fc3 = Linear(16, 2)

    def __call__(self, x):
        # Testing new Tanh operation
        x = self.fc1(x).tanh()
        x = self.fc2(x).tanh()
        x = self.fc3(x)
        return x

    def parameters(self):
        return self.fc1.parameters() + self.fc2.parameters() + self.fc3.parameters()

moon_model = MoonMLP()
moon_optimizer = Adam(moon_model.parameters(), lr=0.01)

epochs = 60
batch_size = 32

for epoch in range(epochs):
    # Shuffle data to prevent the model from memorizing the sequence
    permutation = np.random.permutation(X_moons.shape[0])
    X_shuffled = X_moons[permutation]
    y_shuffled = y_moons[permutation]
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for i in range(0, X_moons.shape[0], batch_size):
        X_batch = Tensor(X_shuffled[i:i+batch_size])
        y_batch = y_shuffled[i:i+batch_size]

        # 1. Forward Pass
        logits = moon_model(X_batch)
        loss = cross_entropy_loss(logits, y_batch)

        # 2. Backward Pass
        moon_model.zero_grad()
        loss.backward()

        # 3. Update Weights
        moon_optimizer.step()

        # Tracking metrics
        epoch_loss += loss.data * len(y_batch)
        predictions = np.argmax(logits.data, axis=1)
        correct += np.sum(predictions == y_batch)
        total += len(y_batch)
        
    if (epoch + 1) % 10 == 0:
        print(f"Moons Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/total:.4f} - Accuracy: {(correct/total)*100:.2f}%")

Moons Epoch 10/60 - Loss: 0.0093 - Accuracy: 99.90%
Moons Epoch 20/60 - Loss: 0.0028 - Accuracy: 100.00%
Moons Epoch 30/60 - Loss: 0.0016 - Accuracy: 100.00%
Moons Epoch 40/60 - Loss: 0.0125 - Accuracy: 99.60%
Moons Epoch 50/60 - Loss: 0.0011 - Accuracy: 100.00%
Moons Epoch 60/60 - Loss: 0.0011 - Accuracy: 100.00%


In [33]:
#fetch minst dataset

In [34]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist["data"], mnist["target"].astype(int)
X = X / 255.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
# train MLP on MINST

In [36]:
class MLP(Module):
    def __init__(self):
        self.fc1 = Linear(784, 128)
        self.relu = ReLU()
        self.fc2 = Linear(128, 10)

    def __call__(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

    def parameters(self):
        return self.fc1.parameters() + self.fc2.parameters()

model = MLP()
optimizer = SGD(model.parameters(), lr=0.1)

epochs = 10
batch_size = 64

for epoch in range(epochs):
    permutation = np.random.permutation(X_train.shape[0])
    X_train_shuffled = X_train[permutation]
    y_train_shuffled = y_train[permutation]
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for i in range(0, X_train.shape[0], batch_size):
        X_batch = Tensor(X_train_shuffled[i:i+batch_size])
        y_batch = y_train_shuffled[i:i+batch_size]
        logits = model(X_batch)
        loss = cross_entropy_loss(logits, y_batch)
        model.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.data * len(y_batch)
        predictions = np.argmax(logits.data, axis=1)
        correct += np.sum(predictions == y_batch)
        total += len(y_batch)
        
    avg_loss = epoch_loss / total
    accuracy = (correct / total) * 100
    print(f"MNIST MLP Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {accuracy:.2f}%")

MNIST MLP Epoch 1/10 - Loss: 0.3792 - Accuracy: 89.38%
MNIST MLP Epoch 2/10 - Loss: 0.2017 - Accuracy: 94.20%


MNIST MLP Epoch 3/10 - Loss: 0.1511 - Accuracy: 95.62%
MNIST MLP Epoch 4/10 - Loss: 0.1207 - Accuracy: 96.61%


MNIST MLP Epoch 5/10 - Loss: 0.1017 - Accuracy: 97.06%
MNIST MLP Epoch 6/10 - Loss: 0.0880 - Accuracy: 97.49%


MNIST MLP Epoch 7/10 - Loss: 0.0772 - Accuracy: 97.82%
MNIST MLP Epoch 8/10 - Loss: 0.0687 - Accuracy: 98.04%


MNIST MLP Epoch 9/10 - Loss: 0.0618 - Accuracy: 98.29%
MNIST MLP Epoch 10/10 - Loss: 0.0562 - Accuracy: 98.50%


In [37]:
# train CNN on MINST

In [38]:
class SimpleCNN(Module):
    def __init__(self):
        # Extracts 4 different visual features from a 1-channel grayscale image
        self.conv1 = Conv2D(in_channels=1, out_channels=4, kernel_size=3, stride=2, padding=1)
        self.flatten = Flatten()
        # The 28x28 image is cut down to 14x14 by the stride of 2
        self.fc1 = Linear(4 * 14 * 14, 10)

    def __call__(self, x):
        x = self.conv1(x)
        x = x.relu() if hasattr(x, 'relu') else ReLU()(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return x

    def parameters(self):
        return self.conv1.parameters() + self.fc1.parameters()

cnn_model = SimpleCNN()
cnn_optimizer = Adam(cnn_model.parameters(), lr=0.005)

# Reshape data back into 4D images: (Batch size, 1 Channel, 28 Height, 28 Width)
X_train_cnn = X_train[:500].reshape(-1, 1, 28, 28)
y_train_cnn = y_train[:500]

epochs = 7
batch_size = 16

for epoch in range(epochs):
    permutation = np.random.permutation(X_train_cnn.shape[0])
    X_shuffled = X_train_cnn[permutation]
    y_shuffled = y_train_cnn[permutation]
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for i in range(0, X_train_cnn.shape[0], batch_size):
        X_batch = Tensor(X_shuffled[i:i+batch_size])
        y_batch = y_shuffled[i:i+batch_size]
        logits = cnn_model(X_batch)
        loss = cross_entropy_loss(logits, y_batch)
        cnn_model.zero_grad()
        loss.backward()
        cnn_optimizer.step()
        
        epoch_loss += loss.data * len(y_batch)
        predictions = np.argmax(logits.data, axis=1)
        correct += np.sum(predictions == y_batch)
        total += len(y_batch)
        
    print(f"CNN Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/total:.4f} - Accuracy: {(correct/total)*100:.2f}%")

CNN Epoch 1/7 - Loss: 1.3450 - Accuracy: 61.60%


CNN Epoch 2/7 - Loss: 0.4447 - Accuracy: 86.60%


CNN Epoch 3/7 - Loss: 0.2623 - Accuracy: 91.80%


CNN Epoch 4/7 - Loss: 0.1559 - Accuracy: 96.60%


CNN Epoch 5/7 - Loss: 0.1010 - Accuracy: 97.60%


CNN Epoch 6/7 - Loss: 0.0789 - Accuracy: 98.00%


CNN Epoch 7/7 - Loss: 0.0433 - Accuracy: 99.80%
